# Adaptive Severity

OPTICS severity audit and score/cluster diagnostics.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in ROOT.parents:
        if (parent / "src").exists():
            ROOT = parent
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.chdir(ROOT)
SAMPLE_ROWS = int(os.getenv("GTD_NOTEBOOK_SAMPLE_ROWS", "25000"))
print(f"Project root: {ROOT}")
print(f"Notebook sample rows: {SAMPLE_ROWS:,}")

Project root: C:\Users\kunmi\Personal Projects\Global-Terrorism-Database
Notebook sample rows: 25,000


In [2]:
import pandas as pd

from gtd_capstone.data.repository import DataRepository

incidents = DataRepository().load_incidents(sample_rows=SAMPLE_ROWS)
severity_cols = [
    "severity",
    "severity_score",
    "severity_score_percentile",
    "severity_cluster",
    "severity_method",
]
incidents[severity_cols].head()

,severity,severity_score,severity_score_percentile,severity_cluster,severity_method
0,Low,1.213008,0.191209,0,adaptive-optics-density
1,None,0.000000,0.000000,-1,none-casualty
2,Low,1.213008,0.191209,0,adaptive-optics-density
3,None,0.000000,0.000000,-1,none-casualty
4,None,0.000000,0.000000,-1,none-casualty


In [3]:
(
    incidents.groupby(["severity", "severity_method"])
    .agg(
        rows=("eventid", "count"),
        median_score=("severity_score", "median"),
        median_casualties=("casualties", "median"),
    )
    .reset_index()
    .sort_values(["median_score", "rows"], ascending=[True, False])
)

,severity,severity_method,rows,median_score,median_casualties
7,None,none-casualty,14562,0.000000,0.0
2,Low,adaptive-optics-density,6799,1.213008,1.0
5,Medium,adaptive-optics-density,980,2.426015,3.0
6,Medium,adaptive-optics-noise-percentile,494,3.172125,7.0
0,High,adaptive-optics-density,651,3.371752,6.0
1,High,adaptive-optics-noise-percentile,750,4.407478,13.0
3,Mass Casualty,adaptive-optics-density,271,4.958123,16.0
4,Mass Casualty,adaptive-optics-noise-percentile,493,6.705455,46.0
